In [0]:
dbutils.library.restartPython()

1. Create two (2) new tables in your own databse where you'll store the predictions from each model for this exercise.

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
-- Create tables in own database
USE gr5069.gbm2118;

In [0]:
%sql
-- Table for storing predictions of positionOrder from results dataset (Model 1)
CREATE TABLE results_ridge_reg (
raceId INT,
circuitId INT,
alt INT,
lng FLOAT,
lat FLOAT,
avg_fastestLapSpeed DOUBLE,
y_pred DOUBLE,
residual DOUBLE
);

In [0]:
%sql
-- Table for storing predictions of point scored from results dataset (Model 2)
CREATE TABLE results_random_forest (
resultId INT,
raceId INT,
driverId INT,
constructorId INT,
grid INT,
points DOUBLE,
y_pred DOUBLE,
residual DOUBLE
);

2. Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments

## Model 1: Ridge Regression

I will predict the average fastest lap speed in km/h using ridge regression. My features are: circuit altitude in meters (alt), circuit longitude (lng) and circuit latitude (lat). I will first convert the target and features, and raceId to ints and floats, depending on their original datatypes, as shown on [mintfly](https://www.mintlify.com/TracingInsights/RaceData/reference/results). I will also convert the fastest lap speed to meters per second so that it is consistent with the circuit altitude.

In [0]:
# Import important libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark
import statsmodels.api as sm
import mlflow.sklearn

In [0]:
# Load the results data
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
df_circuits = spark.read.csv("/Volumes/gr5069/raw/f1_data/circuits.csv", header=True)

# Inspect the data
display(df_results)
display(df_races)
display(df_circuits)

In [0]:
# Replace malformed values with NULLs before casting
from pyspark.sql.functions import col, when, trim

# df_results
df_results = df_results.withColumn(
    "fastestLapSpeed",
    when(trim(col("fastestLapSpeed")) == "\\N", None)
    .otherwise(col("fastestLapSpeed"))
)

# df_circuits
df_circuits = df_circuits.withColumn(
    "lat",
    when(trim(col("lat")) == "\\N", None)
    .otherwise(col("lat"))
)

df_circuits = df_circuits.withColumn(
    "lng",
    when(trim(col("lng")) == "\\N", None)
    .otherwise(col("lng"))
)

df_circuits = df_circuits.withColumn(
    "alt",
    when(trim(col("alt")) == "\\N", None)
    .otherwise(col("alt"))
)

In [0]:
# Convert raceId in df_results to int, and fastestLapSpeed to float 
df_results = df_results.withColumn("fastestLapSpeed", df_results["fastestLapSpeed"].cast("float"))
df_results = df_results.withColumn("raceId", df_results["raceId"].cast("int"))

# Convert raceId and circuitId in df_races to integer
df_races = df_races.withColumn("raceId", df_races["raceId"].cast("int"))
df_races = df_races.withColumn("circuitId", df_races["circuitId"].cast("int"))

# Convert circuitId and alt in df_circuits to integer and lat and lng to floats
df_circuits = df_circuits.withColumn("circuitId", df_circuits["circuitId"].cast("int"))
df_circuits = df_circuits.withColumn("lat", df_circuits["lat"].cast("float"))
df_circuits = df_circuits.withColumn("lng", df_circuits["lng"].cast("float"))
df_circuits = df_circuits.withColumn("alt", df_circuits["alt"].cast("int"))

# Inspect the data
display(df_results)
display(df_races)
display(df_circuits)

In [0]:
# merge df_results and df_races on raceId
df = df_results.join(df_races, on="raceId", how="inner")

# merge df and df_circuits on circuitId
df = df.join(df_circuits, on="circuitId", how="inner")
# Inspect the data
display(df)

In [0]:
# Convert fastestLapSpeed to m/s
df = df.withColumn("fastestLapSpeed", df["fastestLapSpeed"] / 3.6)
display(df)

In [0]:
# Calculate average fastestLapSpeed for each circuitId
df_avg = df.groupBy("circuitId").agg({"fastestLapSpeed": "avg"})
display(df_avg)

In [0]:
#rename avg(fastestLapSpeed) avg_fastestLapSpeed
df_avg = df_avg.withColumnRenamed("avg(fastestLapSpeed)", "avg_fastestLapSpeed")

# Step 2: join back, keeping unique rows
df_grouped = df_avg.join(df, on="circuitId", how="left")

# Step 3: select columns
df_grouped = df_grouped.select(
    "circuitId",
    "raceId",
    "avg_fastestLapSpeed",
    "alt",
    "lng",
    "lat"
)

# Drop duplicates
df_grouped = df_grouped.drop_duplicates(["circuitId", "raceId"])

# Drop Null Values
df_grouped = df_grouped.dropna()

display(df_grouped)

In [0]:
# Change variable names to X, y to prepare for train/test split
df_grouped = df_grouped.toPandas()
y = df_grouped["avg_fastestLapSpeed"]
X = df_grouped[["alt", "lng", "lat"]]

# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Build a ridge regression model

In [0]:
# Linear Regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

params = {"alpha": 1.0}
rr = Ridge(**params)
rr.fit(X_train, y_train.dropna())
predictions = rr.predict(X_test)

# Create metrics to evaluate model
mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
mae_med = median_absolute_error(y_test, predictions)

print("  mse: {}".format(mse))
print("  mae: {}".format(mae))
print("  R2: {}".format(r2))
print("  mae_med: {}".format(mae_med))

# Display coeffiecients
coefficients = pd.DataFrame(rr.coef_, index=X_train.columns, columns=["coefficients"])
coefficients.sort_values("coefficients", ascending=False)
print(coefficients)

# Create plot
fig, ax = plt.subplots()

sns.residplot(x=predictions, y=y_test.astype(float))
plt.xlabel("Predicted values for average fastestLapSpeed in (m/s)")
plt.ylabel("Residual")
plt.title("Residual Plot")